# Run DynaGuide in the Calvin env

## Prerequisites

1. Install the env once with `setup_dynaguide.bat`, see [tutorial_for_pclab.md](../tutorial_for_pclab.md)
1. Download the DynaGuide model from this [link](https://drive.google.com/file/d/1DeOnoDacXjBHgy1DGoRJpYIR8SyDy5fJ/view?usp=drive_link)
1. Download the Base policy from this [link](https://drive.google.com/file/d/1lcI_PBgFIYDsoK4T4qO7SGJJgI2lw0Kd/view?usp=drive_link)
1. Download the Guidance conditions for SWITCH_ON from this [link](https://drive.google.com/file/d/1wtEGnG87Y-imqD2MygcqbJwp_RGi7YNB/view?usp=drive_link)
1. Put these three downloaded files in the folder _DynaGuide-main_, keeping their names. `calvin_exp_configs_examples/switch_on.json` already points to the guidance condition there, so nothing needs editing

## Running DynaGuide rollouts

Now, run this command in the windows, Anaconda Prompt. The `^` at the end of a line continues the command on the next line, like `\` on Linux. Nothing may follow it, not even a space, and the last line has none. While pasting, the prompt shows `More?` for each continued line, this is normal. You can also past the code in a .bat file and execute this.
```bat
conda activate dynaguide
cd DynaGuide-main
python run_dynaguide.py ^
    --output_folder results\switch_on_test ^
    --agent base_policy_300.pth ^
    --guidance dynamics_model_5900.pth ^
    --exp_setup_config calvin_exp_configs_examples\switch_on.json ^
    --horizon 400 ^
    --n_rollouts 3 ^
    --video_skip 2 ^
    --camera_names third_person ^
    --scale 1.5 ^
    --ss 4 ^
    --alpha 30 ^
    --save_frames
```

The paths are relative, so it has to run from inside `DynaGuide-main`. A PyBullet window opens during the rollouts, leave it open.

Everything of the run lands in `--output_folder`, which is created if it doesn't exist: `<date>_<time>_video.mp4`, `<date>_<time>_rollout.hdf5`, `<date>_<time>_log.json`, `args.json` and the frames from `--save_frames`. Use a new folder for each run. If the folder already holds results, the run stops right away so nothing gets overwritten.

**Faster.** This has the same physics and the same images for the policy, it only changes what you see. The video is still saved:
```bat
python run_dynaguide.py ^
    --output_folder results\switch_on_fast ^
    --agent base_policy_300.pth ^
    --guidance dynamics_model_5900.pth ^
    --exp_setup_config calvin_exp_configs_examples\switch_on.json ^
    --horizon 400 ^
    --n_rollouts 3 ^
    --video_skip 4 ^
    --camera_names third_person ^
    --scale 1.5 ^
    --ss 4 ^
    --alpha 30 ^
    --no_live_render ^
    --no_collage
```

| Option | What it does | What you lose |
|---|---|---|
| `--no_live_render` | The PyBullet window stops redrawing | The window stays open but frozen, watch the video afterwards |
| `--no_collage` | Skips the dynamics model reconstructions | The tiles above the video and in `--save_frames` |
| `--video_skip 4` | Saves every 4th step instead of every 2nd | The video is choppier and shorter |

`Time_Policy`, `Time_Env`, `Time_Collage` and `Time_Video` in the log show the average seconds per rollout for each part, and `time` the total. `Time_Policy` is the guidance itself and can't be reduced without changing DynaGuide.

To get more rollouts per hour, you can also start a second run in another Anaconda Prompt with its own `--output_folder` and `--seed`. The GTX 1650 has 4 GB of memory, so check with `nvidia-smi` that both fit.

## Evaluating success automatically

CALVIN has no success signal of its own, so `core/calvin_behaviors.py` detects what the robot achieved from the states and compares this to the goal in the experiment config. You don't need a separate analysis step or config. After the run you can watch the video and read the result in the terminal and in `<date>_<time>_log.json` in the output folder:

```json
"Target_Behaviors": ["switch_on"],
"Avoid_Behaviors": [],
"Success_Rate": 0.6667,
"Num_Success": 2,
"Num_Episode": 3,
"Behavior_Counts": {"switch_on": 2, "door_left": 1},
"Episode_Behaviors": [["switch_on"], ["door_left"], ["switch_on"]]
```

- `Success_Rate`, `Num_Success`: the rollouts that reached the goal. Both are `null` if the config has no goal, like `control.json`
- `Behavior_Counts`: how often each behavior happened. This also tells you what the robot did instead when it failed
- `Episode_Behaviors`: the behavior of each rollout, in order, so you can look up a specific rollout in the video

### Scoring a file you already have

This needs no rollouts. Pass the output folder of the run, or a single hdf5 file, which needs `--dataset_obs` if it comes from another script. Files from `run_trained_agent.py`, `run_sampling_baseline.py` and older `run_dynaguide.py` runs stop at the first small change, usually before the goal is reached, so they mostly score `no_behavior`:

```bat
python evaluate_behaviors.py ^
    --dataset results\switch_on_test ^
    --config calvin_exp_configs_examples\switch_on.json ^
    --output results\switch_on_test\behaviors.json
```

### Running other tasks

Pass a different config with `--exp_setup_config` and download the matching guidance condition. All configs in `calvin_exp_configs_examples` already have their goal, so only `pos_examples` / `neg_examples` need to point to your files. A goal looks like this:

```json
"target_behaviors" : ["switch_on"],
```

| Field | Meaning | Example config |
|---|---|---|
| `target_behaviors` | Success if **any** of these happens | `switch_on.json`, `multiple_and_negative_objectives/button_on_switch_on.json` |
| `avoid_behaviors` | Failure if any of these happens | `door_dont_right.json` |
| both | A target happened and nothing to avoid happened | |
| neither | Only the behavior counts are reported | `control.json` |

With only `avoid_behaviors`, a rollout where the robot does nothing counts as a success, so check `Behavior_Counts` for `no_behavior`. A typo in a behavior name stops the run with an error before the rollouts start.

### When a rollout ends

The config decides this, `run_dynaguide.py` only asks `GoalTracker` in `core/calvin_behaviors.py` whether the outcome is decided. A rollout ends when one of these happens:

- A behavior from `target_behaviors` is reached (success)
- A behavior from `avoid_behaviors` happens (failure)
- `--horizon` steps have passed. This is the only way to end for `control.json`, and for avoid-only configs it means success

Doing something else does not end the rollout: if the robot opens the drawer on the way to the switch, it continues and can still reach the switch. `Episode_Behaviors` lists everything it achieved, in order. So set `--horizon` high enough for the task, and expect failed rollouts to take the full time.

These results aren't comparable to the paper's numbers. The paper stops at the first small change and measures which object was touched first, while here the robot keeps trying until it reaches the goal. To compare with the paper, use the `sanity_` behaviors, see "Replicating the paper" below. For a baseline, run the same config with `--scale 0 --ss 1`.

### Behaviors and thresholds

A behavior counts when the state goes from the start range to the end range, measured from the first state of the rollout. These are the end states from `data_processing_calvin/calvin_to_labeled_hdf5.py`, the authors' definition of a behavior in the guidance data:

| Behavior | Start | End | Range in the simulator |
|---|---|---|---|
| `door_left` / `door_right` | below 0.05 / above 0.25 | above 0.25 / below 0.05 | 0 to 0.27 |
| `drawer_open` / `drawer_close` | below 0.05 / above 0.10 | above 0.10 / below 0.05 | 0 to 0.16 |
| `switch_on` / `switch_off` | below 0.01 / above 0.07 | above 0.07 / below 0.01 | 0 to 0.09, the light changes at about 0.045 |
| `button_on` / `button_off` | below 0.2 / above 0.8 | above 0.8 / below 0.2 | 0 or 1 |
| `red_displace`, `blue_displace`, `pink_displace` | anywhere | moved more than 0.01 with the gripper within 0.1 | |
| `no_behavior` | | none of these happened | |

The block condition is the weakest one, 0.01 is only a small push. The block labels mean the block was moved, not necessarily lifted, even though the configs are called `red_lift` etc.

To make a goal stricter, change its numbers in `ARTICULATED` or `BLOCK_MOVED` in `core/calvin_behaviors.py`. Evaluation and termination both follow, because they use the same code. To add a new behavior, add its name to `BEHAVIORS` and its check to `GoalTracker.detect`. Then it can be used in any config.

The cell below shows the results of a run. Set the folder and run it from a kernel started in `DynaGuide-main`.

In [ ]:
import glob
import json

output_folder = "results/switch_on_test"
with open(glob.glob(f"{output_folder}/*_log.json")[0], "r") as f:
    results = json.load(f)

print(f"Goal: {results['Target_Behaviors']}, avoid: {results['Avoid_Behaviors']}")
print(f"Success: {results['Num_Success']} / {results['Num_Episode']}")
print(f"Behaviors: {results['Behavior_Counts']}")
for i, behaviors in enumerate(results["Episode_Behaviors"]):
    print(f"Rollout {i}: {behaviors}")

## Experiment configs

An experiment config is the json file you pass with `--exp_setup_config`. The examples are in `calvin_exp_configs_examples`. To make your own, copy one and change these fields:

```json
{
    "target_behaviors" : ["switch_on"],
    "env_setup" : {"switch" : 0},
    "use_neg" : false,
    "pos_examples": "guidance_condition_switch_on_20.hdf5",
    "reset_poses": "calvin_exp_configs_examples/reset_poses/initial_calvin_robot_states_right_side_midpoint.json"
}
```

| Field | What you can write | Meaning |
|---|---|---|
| `target_behaviors` | A list of behaviors, see below | Success if any of them happens, and the rollout ends. Optional |
| `avoid_behaviors` | A list of behaviors, see below | Failure if any of them happens, and the rollout ends. Optional |
| `env_setup` | `sliding_door`, `drawer`, `switch`, `green_light` with a start value, and `cubes_table_only` as `true` | The start of each rollout. Everything left out is randomly open or closed, on or off. Must allow the goal: for `switch_on`, the switch has to start at 0 |
| `use_neg` | `true` or `false` | Whether to steer away from `neg_examples` |
| `pos_examples` | Path to an hdf5 file, or `null` | Guidance conditions to steer towards. Relative paths start from `DynaGuide-main` |
| `neg_examples` | Path to an hdf5 file, or `null` | Guidance conditions to steer away from. Only used with `"use_neg": true` |
| `loc_target` | Three numbers, x y z | Target position, only used with `--direct_position_guidance` |
| `reset_poses` | Path to a json file, or `null` | Robot start poses, one is picked randomly per rollout. The files are in `calvin_exp_configs_examples/reset_poses` |

The start ranges for `env_setup` are `sliding_door` 0 to 0.27, `drawer` 0 to 0.16, `switch` 0 to 0.09 and `green_light` 0 or 1.

**Behaviors you can use as goals:**
- `button_on`, `button_off`, `switch_on`, `switch_off`, `drawer_open`, `drawer_close`, `door_left`, `door_right`
- `red_displace`, `blue_displace`, `pink_displace`
- The same names with `sanity_` in front, like `sanity_switch_on`, and `sanity_any`

**Examples:**

```json
"target_behaviors" : ["drawer_open"],
```
```json
"target_behaviors" : ["button_on", "switch_on"],
```
```json
"avoid_behaviors" : ["drawer_open"],
```
```json
"target_behaviors" : ["sanity_switch_on"],
```

The goal only decides success and when the rollout ends. It doesn't steer the robot, that is the job of `pos_examples` and `neg_examples`, so they should show the same behavior as the goal.

## Replicating the paper

The `sanity_` behaviors replicate the paper's numbers instead of checking that the task is finished. Using one switches the whole rollout to the paper's rules:

| | Normal behaviors | `sanity_` behaviors |
|---|---|---|
| Rollout ends | At the goal, or at `--horizon` | At the first small change of **anything**, or at `--horizon` |
| Success | The end state from the table above is reached | The first interaction was the target |
| Example, switch | Above 0.07 | Moved by 0.02, the rollout ends at 0.045 |
| Example, block | Moved 0.01 | Moved 0.001 with the gripper within 0.1 |
| Code | `GoalTracker` | `SanityTracker`, using the authors' `check_state_difference` and the labels of `analyze_calvin_touch.py` |

Normal and `sanity_` behaviors can't be mixed in one config. `sanity_any` counts any interaction as a success, for the control runs.

Read the result from `Full_Distribution` in the results json. This is the number the paper plots, the share of each first interaction. It can differ slightly from `Success_Rate` when the robot moves several blocks at once.

**1. DynaGuide**, with `calvin_exp_configs_examples\sanity_switch_on.json`:
```bat
python run_dynaguide.py ^
    --output_folder results\sanity_switch_on ^
    --agent base_policy_300.pth ^
    --guidance dynamics_model_5900.pth ^
    --exp_setup_config calvin_exp_configs_examples\sanity_switch_on.json ^
    --horizon 400 ^
    --n_rollouts 50 ^
    --video_skip 10 ^
    --camera_names third_person ^
    --scale 1.5 ^
    --ss 4 ^
    --alpha 30 ^
    --no_live_render ^
    --no_collage
```

**2. Base policy without guidance**, on the same config. Use a new `--output_folder`, like `results\sanity_switch_on_base`, and `--scale 0 --ss 1` without `--alpha`.

**3. Control**, the base policy with a random start. Use a new `--output_folder`, `calvin_exp_configs_examples\sanity_control.json` and `--scale 0 --ss 1`.

**Settings from `paper_experiments/calvin_articulated_object.sh` and `calvin_movable_object.sh`.** All use `--ss 4 --horizon 400 --n_rollouts 50`:

| Task | `--scale` | `--alpha` |
|---|---|---|
| `switch_on`, `switch_off`, `door_left` | 1.5 | 30 |
| `button_on`, `button_off` | 1 | 30 |
| `drawer_open`, `drawer_close` | 1 | 40 |
| `door_right` | 2 | 15 |
| `red_lift`, `blue_lift`, `pink_lift` | 2 | 30 |

For another task, copy its config and write its `sanity_` name in `target_behaviors`. For `red_lift`, that is `sanity_red_displace`.

To score rollouts you already have by the paper's rules, pass a `sanity_` config to `evaluate_behaviors.py`.

**What still differs from the paper:**
- The paper averages 6 base policies with 50 rollouts each. We have one downloaded base policy, so expect more spread
- Only the `SWITCH_ON` guidance condition can be downloaded, the others have to be built from the CALVIN dataset
- The paper's images came from Linux with the EGL renderer. On Windows the camera uses the CPU TinyRenderer, whose images look different from the training images, so a lower score on Windows doesn't have to mean a bug


To check the guidance conditions, run:
```bat
python helper_scripts\show_guidance_images.py ^
    --dataset guidance_condition_switch_on_20.hdf5 ^
    --output results\guidance_images
```